ONECALL MODEL 1: ALL-IN-ONE COLAB PIPELINE (ZERO LOCAL UPLOAD REQUIRED)

In [1]:
# ==============================================================================
# CELL 1: Install Dependencies (Pinned for Binary Stability)
# ==============================================================================
!pip install --quiet "numpy<2.0.0" pandas pyarrow


In [2]:
# ==============================================================================
# CELL 2: Imports, Environment Configuration & Compatibility Setup
# ==============================================================================
import os
import re
import json
import time
from pathlib import Path
from contextlib import contextmanager
import numpy as np

if not hasattr(np, "_no_nep50_warning"):
    @contextmanager
    def _no_nep50_warning():
        yield
    np._no_nep50_warning = _no_nep50_warning

import pandas as pd
from huggingface_hub import list_repo_files, hf_hub_download

import pandas.errors
if not hasattr(pandas.errors, "Pandas4Warning"):
    class Pandas4Warning(UserWarning):
        pass
    pandas.errors.Pandas4Warning = Pandas4Warning

# Path resolution: local ./content if running locally, /content if running in Colab
if (Path("./content") / "onecall_model1_combined_no_severity.csv").exists() or Path("./content").exists():
    BASE_DIR = Path("./content")
elif Path("/content").exists():
    BASE_DIR = Path("/content")
else:
    BASE_DIR = Path("./content")

BASE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = BASE_DIR
print(f"[Environment Ready] Destination Output Directory: {OUTPUT_DIR.resolve()}")


In [3]:
# ==============================================================================
# CELL 3: Robust Helper Functions (Pure Dictionary Contingency, Safe Download)
# ==============================================================================
EXCLUDED_PATTERNS = [
    r"dataset_card\.json$",
    r"dataset_info\.json$",
    r"README\.md$",
    r"^\.git",
    r"LICENSE",
    r"metadata",
    r"\.txt$"
]

def find_dataset_files(repo_id: str) -> list:
    """Discovers all non-metadata data files in a Hugging Face repository."""
    all_files = list_repo_files(repo_id=repo_id, repo_type="dataset")
    data_files = []
    for f in all_files:
        if any(re.search(pat, f, re.IGNORECASE) for pat in EXCLUDED_PATTERNS):
            continue
        if f.lower().endswith((".csv", ".json", ".jsonl", ".parquet", ".tsv")):
            data_files.append(f)
    if not data_files:
        raise FileNotFoundError(f"No valid data files found in repository '{repo_id}'. Found: {all_files}")
    return sorted(data_files)

def download_with_retry(repo_id: str, filename: str, max_retries: int = 3) -> str:
    """Downloads a file from Hugging Face Hub with exponential backoff retries."""
    for attempt in range(1, max_retries + 1):
        try:
            return hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset")
        except Exception as e:
            if attempt == max_retries:
                raise RuntimeError(f"Failed to download {filename} from {repo_id} after {max_retries} attempts: {e}")
            time.sleep(2 * attempt)

def load_raw_file(file_path: str) -> pd.DataFrame:
    """Reads a downloaded data file into a Pandas DataFrame based on extension."""
    path_str = str(file_path).lower()
    if path_str.endswith(".parquet"):
        return pd.read_parquet(file_path)
    elif path_str.endswith(".jsonl"):
        records = []
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    records.append(json.loads(line))
        return pd.json_normalize(records)
    elif path_str.endswith(".json"):
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return pd.json_normalize(data) if isinstance(data, list) else pd.DataFrame([data])
    elif path_str.endswith((".csv", ".tsv")):
        sep = "\t" if path_str.endswith(".tsv") else ","
        return pd.read_csv(file_path, sep=sep)
    else:
        raise ValueError(f"Unsupported file format: {file_path}")

def compute_contingency_table(df: pd.DataFrame, row_col: str, col_col: str) -> pd.DataFrame:
    """
    Computes a 2D contingency distribution using pure Python dictionary mapping.
    Completely avoids pd.crosstab(), unstack(), pivot(), and pandas.core.reshape.
    """
    row_vals = sorted([r for r in df[row_col].dropna().unique().tolist() if str(r).strip() != ""])
    col_vals = sorted([c for c in df[col_col].dropna().unique().tolist() if str(c).strip() != ""])

    matrix = {}
    for r in row_vals:
        matrix[r] = {}
        sub_df = df[df[row_col] == r]
        for c in col_vals:
            matrix[r][c] = int((sub_df[col_col] == c).sum())

    return pd.DataFrame.from_dict(matrix, orient="index")

def normalize_text_series(series: pd.Series) -> pd.Series:
    """Standardizes string values: strips leading/trailing whitespace and collapses internal spaces."""
    return series.fillna("").astype(str).str.strip().str.replace(r"\s+", " ", regex=True)


In [4]:
# ==============================================================================
# CELL 4: Dataset 1 Loading & Inspection (syntech-ai/medical-triage-500)
# ==============================================================================
REPO_1 = "syntech-ai/medical-triage-500"
print(f"[Dataset 1] Inspecting repository: {REPO_1}")

repo1_files = find_dataset_files(REPO_1)
print(f"[Dataset 1] Identified valid data files: {repo1_files}")

# Prefer JSONL to preserve nested structure, otherwise CSV/Parquet
selected_file_1 = next((f for f in repo1_files if f.endswith(".jsonl")), repo1_files[0])
print(f"[Dataset 1] Selected file for download: {selected_file_1}")

local_path_1 = download_with_retry(REPO_1, selected_file_1)
raw_syntech_df = load_raw_file(local_path_1)

print(f"[Dataset 1] Raw dimensions: {raw_syntech_df.shape[0]} rows, {raw_syntech_df.shape[1]} columns")
print(f"[Dataset 1] Raw columns detected:\n{raw_syntech_df.columns.tolist()[:15]}")


[Dataset 1] Inspecting repository: syntech-ai/medical-triage-500
[Dataset 1] Identified valid data files: ['medical_triage_500.csv', 'medical_triage_500.jsonl']
[Dataset 1] Selected file for download: medical_triage_500.jsonl


[Dataset 1] Raw dimensions: 500 rows, 13 columns
[Dataset 1] Raw columns detected:
['case_id', 'safety_notice', 'patient.age', 'patient.gender', 'presentation.symptoms', 'presentation.symptom_count', 'presentation.duration', 'presentation.onset', 'presentation.context', 'risk_assessment.risk_level', 'risk_assessment.red_flags', 'triage_classification.urgency_category', 'triage_classification.urgency_reasoning']


In [5]:
# ==============================================================================
# CELL 5: Dataset 1 Flattening & Mapping to Exact OneCall Schema
# ==============================================================================
syntech_mapped = pd.DataFrame()

# 1. id = case_id
syntech_mapped["id"] = raw_syntech_df["case_id"].astype(str)

# 2. symptoms & patient_query = presentation.symptoms converted to text
def extract_symptoms_text(val):
    if isinstance(val, (list, np.ndarray)):
        return ", ".join([str(x).strip() for x in val if str(x).strip()])
    elif pd.notna(val):
        return str(val).strip()
    return ""

symptoms_series = raw_syntech_df["presentation.symptoms"].apply(extract_symptoms_text) if "presentation.symptoms" in raw_syntech_df.columns else raw_syntech_df["presentation_symptoms"].apply(extract_symptoms_text)
syntech_mapped["symptoms"] = symptoms_series
syntech_mapped["patient_query"] = symptoms_series

# 3. severity = risk_level
syntech_mapped["severity"] = raw_syntech_df["risk_assessment.risk_level"] if "risk_assessment.risk_level" in raw_syntech_df.columns else raw_syntech_df.get("risk_level", "")
syntech_mapped["severity"] = syntech_mapped["severity"].fillna("").astype(str).str.lower().str.strip()

# 4. duration = presentation.duration
syntech_mapped["duration"] = raw_syntech_df["presentation.duration"] if "presentation.duration" in raw_syntech_df.columns else raw_syntech_df.get("duration", "")
syntech_mapped["duration"] = syntech_mapped["duration"].fillna("").astype(str).str.strip()

# 5. description = presentation.context
syntech_mapped["description"] = raw_syntech_df["presentation.context"] if "presentation.context" in raw_syntech_df.columns else raw_syntech_df.get("context", "")
syntech_mapped["description"] = syntech_mapped["description"].fillna("").astype(str).str.strip()

# 6. age = patient.age (numeric, NaN preserved)
age_col = "patient.age" if "patient.age" in raw_syntech_df.columns else "age"
syntech_mapped["age"] = pd.to_numeric(raw_syntech_df[age_col], errors="coerce")

# 7. gender = patient.gender
gender_col = "patient.gender" if "patient.gender" in raw_syntech_df.columns else "gender"
syntech_mapped["gender"] = raw_syntech_df[gender_col].fillna("").astype(str).str.lower().str.strip()

# 8. comorbidities = empty string if unavailable (do NOT fabricate)
syntech_mapped["comorbidities"] = ""

# 9. urgency = triage_classification.urgency_category
urgency_col = "triage_classification.urgency_category" if "triage_classification.urgency_category" in raw_syntech_df.columns else "urgency_category"
syntech_mapped["urgency"] = raw_syntech_df[urgency_col].fillna("").astype(str).str.lower().str.strip()

# 10. source_dataset
syntech_mapped["source_dataset"] = "SynTech_medical_triage_500"

# Enforce exact OneCall schema order
ONECALL_SCHEMA = [
    "id", "patient_query", "symptoms", "severity", "duration",
    "description", "age", "gender", "comorbidities", "urgency", "source_dataset"
]
syntech_mapped = syntech_mapped[ONECALL_SCHEMA]

print("[Dataset 1] Mapped successfully to OneCall schema.")
print(f"Sample row:\n{syntech_mapped.iloc[0].to_dict()}")


[Dataset 1] Mapped successfully to OneCall schema.
Sample row:
{'id': 'MTG-00001', 'patient_query': 'blurred vision, chest pain', 'symptoms': 'blurred vision, chest pain', 'severity': 'high', 'duration': '1 week', 'description': 'at rest', 'age': 66, 'gender': 'male', 'comorbidities': '', 'urgency': 'immediate', 'source_dataset': 'SynTech_medical_triage_500'}


In [6]:
# ==============================================================================
# CELL 6: Dataset 2 Loading & Inspection (sweatSmile/medical-symptom-triage-csv)
# ==============================================================================
REPO_2 = "sweatSmile/medical-symptom-triage-csv"
print(f"[Dataset 2] Inspecting repository: {REPO_2}")

repo2_files = find_dataset_files(REPO_2)
print(f"[Dataset 2] Identified valid data files: {repo2_files}")

# Download all partitions (e.g., train and validation parquets)
downloaded_dfs = []
for f in repo2_files:
    print(f"  Downloading partition: {f}")
    local_p = download_with_retry(REPO_2, f)
    part_df = load_raw_file(local_p)
    downloaded_dfs.append(part_df)

raw_sweatsmile_df = pd.concat(downloaded_dfs, ignore_index=True) if len(downloaded_dfs) > 1 else downloaded_dfs[0]

print(f"[Dataset 2] Raw dimensions combined: {raw_sweatsmile_df.shape[0]} rows, {raw_sweatsmile_df.shape[1]} columns")
print(f"[Dataset 2] Raw columns detected:\n{raw_sweatsmile_df.columns.tolist()}")
print(f"[Dataset 2] Data types:\n{raw_sweatsmile_df.dtypes}")


[Dataset 2] Inspecting repository: sweatSmile/medical-symptom-triage-csv
[Dataset 2] Identified valid data files: ['data/train-00000-of-00001.parquet', 'data/validation-00000-of-00001.parquet']
[Dataset 2] Raw dimensions combined: 1390 rows, 10 columns
[Dataset 2] Raw columns detected:
['id', 'symptom_description', 'demographic_context', 'duration', 'primary_specialty', 'urgency_level', 'confidence_score', 'critical_questions', 'possible_conditions', 'reasoning']
[Dataset 2] Data types:
id                     int64
symptom_description      str
demographic_context      str
duration                 str
primary_specialty        str
urgency_level            str
confidence_score         str
critical_questions       str
possible_conditions      str
reasoning                str
dtype: object


In [7]:
# ==============================================================================
# CELL 7: Dataset 2 Automatic Column Detection & Mapping to OneCall Schema
# ==============================================================================
col_lookup = {c.lower(): c for c in raw_sweatsmile_df.columns}

# Automatic column identification heuristics
query_candidate = next((col_lookup[c] for c in ["symptom_description", "query", "patient_query", "symptoms", "complaint", "text"] if c in col_lookup), None)
urgency_candidate = next((col_lookup[c] for c in ["urgency_level", "urgency", "triage", "target", "label"] if c in col_lookup), None)
duration_candidate = next((col_lookup[c] for c in ["duration", "onset", "time"] if c in col_lookup), None)
desc_candidate = next((col_lookup[c] for c in ["reasoning", "description", "context", "notes", "possible_conditions"] if c in col_lookup), None)
id_candidate = next((col_lookup[c] for c in ["id", "case_id", "patient_id"] if c in col_lookup), None)
demographics_candidate = next((col_lookup[c] for c in ["demographic_context", "demographics", "patient_info"] if c in col_lookup), None)

print(f"[Dataset 2] Column Detection Results:")
print(f"  ID: {id_candidate} | Query: {query_candidate} | Urgency: {urgency_candidate}")
print(f"  Duration: {duration_candidate} | Demographics: {demographics_candidate} | Description: {desc_candidate}")

sweatsmile_mapped = pd.DataFrame()

# 1. id
if id_candidate:
    sweatsmile_mapped["id"] = raw_sweatsmile_df[id_candidate].astype(str)
else:
    sweatsmile_mapped["id"] = ["SMILE-" + str(i + 1).zfill(5) for i in range(len(raw_sweatsmile_df))]

# 2. patient_query & symptoms
text_series = normalize_text_series(raw_sweatsmile_df[query_candidate])
sweatsmile_mapped["patient_query"] = text_series
sweatsmile_mapped["symptoms"] = text_series

# 3. severity: None exists in sweatSmile, do NOT fabricate (preserved as empty string)
sweatsmile_mapped["severity"] = ""

# 4. duration
if duration_candidate:
    sweatsmile_mapped["duration"] = normalize_text_series(raw_sweatsmile_df[duration_candidate])
else:
    sweatsmile_mapped["duration"] = ""

# 5. description
if desc_candidate:
    sweatsmile_mapped["description"] = normalize_text_series(raw_sweatsmile_df[desc_candidate])
else:
    sweatsmile_mapped["description"] = ""

# 6, 7, 8: Parse demographics context if present (extract genuine age, gender, comorbidities without fabrication)
extracted_age = []
extracted_gender = []
extracted_comorb = []

if demographics_candidate:
    for val in raw_sweatsmile_df[demographics_candidate].fillna("").astype(str):
        # Extract age: e.g., '58-year-old', '58 yo', 'age 58'
        age_match = re.search(r"(\d{1,3})\s*(?:[-–—]|\s)?(?:years?(?:\s*old)?|yo)\b", val, re.IGNORECASE)
        if age_match:
            extracted_age.append(float(age_match.group(1)))
        else:
            extracted_age.append(np.nan)

        # Extract gender: male/female
        gender_match = re.search(r"\b(female|male|woman|man)\b", val, re.IGNORECASE)
        if gender_match:
            g_str = gender_match.group(1).lower()
            extracted_gender.append("female" if g_str in ["female", "woman"] else "male")
        else:
            extracted_gender.append("")

        # Extract comorbidities / medical history if noted after age/gender
        hist_match = re.search(r"(?:with\s+history\s+of|history\s+of|hx\s+of|known\s+case\s+of)\s+([^,;]+)", val, re.IGNORECASE)
        if hist_match:
            extracted_comorb.append(hist_match.group(1).strip())
        elif "," in val:
            parts = [p.strip() for p in val.split(",")[1:] if p.strip()]
            extracted_comorb.append("; ".join(parts) if parts else "")
        else:
            extracted_comorb.append("")
else:
    extracted_age = [np.nan] * len(raw_sweatsmile_df)
    extracted_gender = [""] * len(raw_sweatsmile_df)
    extracted_comorb = [""] * len(raw_sweatsmile_df)

sweatsmile_mapped["age"] = extracted_age
sweatsmile_mapped["gender"] = extracted_gender
sweatsmile_mapped["comorbidities"] = extracted_comorb

# 9. urgency
sweatsmile_mapped["urgency"] = raw_sweatsmile_df[urgency_candidate].fillna("").astype(str).str.lower().str.strip()

# 10. source_dataset
sweatsmile_mapped["source_dataset"] = "medical_symptom_triage_csv"

# Enforce exact OneCall schema order
sweatsmile_mapped = sweatsmile_mapped[ONECALL_SCHEMA]

print("[Dataset 2] Mapped successfully to OneCall schema.")
print(f"Sample row:\n{sweatsmile_mapped.iloc[0].to_dict()}")


[Dataset 2] Column Detection Results:
  ID: id | Query: symptom_description | Urgency: urgency_level
  Duration: duration | Demographics: demographic_context | Description: reasoning
[Dataset 2] Mapped successfully to OneCall schema.
Sample row:
{'id': '1', 'patient_query': "I've been having crushing chest pain in the center of my chest for the last hour. It feels like an elephant is sitting on my chest. The pain is radiating down my left arm and up into my jaw. I'm also very sweaty, nauseous, and short of breath. I took some antacids thinking it was heartburn but it didn't help at all. I've never felt anything like this before.", 'symptoms': "I've been having crushing chest pain in the center of my chest for the last hour. It feels like an elephant is sitting on my chest. The pain is radiating down my left arm and up into my jaw. I'm also very sweaty, nauseous, and short of breath. I took some antacids thinking it was heartburn but it didn't help at all. I've never felt anything like 

In [8]:
# ==============================================================================
# CELL 8: Label Inspection & Semantic Normalization
# ==============================================================================
print("=" * 80)
print("LABEL AUDIT & PRE-COMBINATION NORMALIZATION")
print("=" * 80)

print("Original Urgency Labels in SynTech:")
print(syntech_mapped["urgency"].value_counts(dropna=False))

print("\nOriginal Urgency Labels in sweatSmile:")
print(sweatsmile_mapped["urgency"].value_counts(dropna=False))

# Semantic mapping into OneCall 3-tier target: low / medium / high
SYNTECH_LABEL_MAP = {
    "immediate": "high",
    "emergency": "high",
    "urgent": "medium",
    "routine": "low",
    "non-urgent": "low",
    "low": "low",
    "medium": "medium",
    "high": "high"
}

SWEATSMILE_LABEL_MAP = {
    "emergency": "high",
    "urgent": "medium",
    "observation": "medium",   # Observation represents intermediate clinic/monitoring care
    "routine": "low",
    "low": "low",
    "medium": "medium",
    "high": "high"
}

syntech_mapped["urgency_clean"] = syntech_mapped["urgency"].map(SYNTECH_LABEL_MAP)
sweatsmile_mapped["urgency_clean"] = sweatsmile_mapped["urgency"].map(SWEATSMILE_LABEL_MAP)

# Report unmapped values if any
unmapped_syn = syntech_mapped[syntech_mapped["urgency_clean"].isna()]["urgency"].unique()
unmapped_sw = sweatsmile_mapped[sweatsmile_mapped["urgency_clean"].isna()]["urgency"].unique()

if len(unmapped_syn) > 0:
    print(f"[WARNING] Unmapped SynTech labels: {unmapped_syn}")
if len(unmapped_sw) > 0:
    print(f"[WARNING] Unmapped sweatSmile labels: {unmapped_sw}")

syntech_mapped["urgency"] = syntech_mapped["urgency_clean"]
sweatsmile_mapped["urgency"] = sweatsmile_mapped["urgency_clean"]

syntech_mapped.drop(columns=["urgency_clean"], inplace=True)
sweatsmile_mapped.drop(columns=["urgency_clean"], inplace=True)

print("\nNormalized Urgency Distribution in SynTech:")
print(syntech_mapped["urgency"].value_counts(dropna=False))

print("\nNormalized Urgency Distribution in sweatSmile:")
print(sweatsmile_mapped["urgency"].value_counts(dropna=False))


LABEL AUDIT & PRE-COMBINATION NORMALIZATION
Original Urgency Labels in SynTech:
urgency
immediate    230
urgent       195
routine       75
Name: count, dtype: int64

Original Urgency Labels in sweatSmile:
urgency
routine        694
urgent         472
emergency      112
observation    112
Name: count, dtype: int64

Normalized Urgency Distribution in SynTech:
urgency
high      230
medium    195
low        75
Name: count, dtype: int64

Normalized Urgency Distribution in sweatSmile:
urgency
low       694
medium    584
high      112
Name: count, dtype: int64


In [9]:
# ==============================================================================
# CELL 9: Data Quality & Target Leakage Audit (Pure Dictionary - No Reshape)
# ==============================================================================
print("=" * 80)
print("DATA QUALITY & TARGET LEAKAGE AUDIT")
print("=" * 80)

def audit_dataset(df: pd.DataFrame, dataset_name: str):
    print(f"\n--- AUDIT: {dataset_name} ---")
    print(f"Total Rows: {len(df)}")
    print(f"Columns: {df.columns.tolist()}")

    print("\nMissing Values:")
    print(df.isna().sum()[df.isna().sum() > 0])

    exact_dups = df.duplicated().sum()
    print(f"Exact Duplicate Rows: {exact_dups}")

    q_dups = df.duplicated(subset=["patient_query"]).sum()
    print(f"Duplicate Queries: {q_dups} ({q_dups / len(df) * 100:.2f}%)")

    print("\nUrgency Distribution:")
    print(df["urgency"].value_counts(dropna=False))

    print("\nSeverity Distribution:")
    print(df["severity"].value_counts(dropna=False))

    dur_avail = (df["duration"].fillna("").astype(str).str.strip() != "").mean() * 100
    age_avail = df["age"].notna().mean() * 100
    print(f"Duration Availability: {dur_avail:.2f}%")
    print(f"Age Availability: {age_avail:.2f}%")

    # Severity-to-Urgency relationship calculated without pd.crosstab() or unstack()
    valid_sev_df = df[df["severity"].fillna("").astype(str).str.strip() != ""]
    if len(valid_sev_df) > 0:
        contingency = compute_contingency_table(valid_sev_df, "severity", "urgency")
        print("\nSeverity vs Urgency Contingency Count (Pure Dictionary Calculation):")
        print(contingency)

        # Row-normalized conditional probability P(urgency | severity)
        row_totals = contingency.sum(axis=1)
        cond_prob = contingency.div(row_totals, axis=0)
        print("\nConditional Probability P(urgency | severity):")
        print(cond_prob.round(4))

        # Check deterministic shortcut
        max_prob_per_row = cond_prob.max(axis=1)
        if (max_prob_per_row == 1.0).all():
            print("\n🚨 CRITICAL LEAKAGE FLAG: Severity deterministically encodes urgency (1-to-1 shortcut).")
            print("   Severity MUST be excluded in genuine baseline modeling to prevent artificial 100% accuracy.")
        else:
            print("\n[OK] Severity does not perfectly encode urgency deterministically.")
    else:
        print("\n[Notice] Severity column is empty in this dataset; no leakage present.")

audit_dataset(syntech_mapped, "SynTech (medical-triage-500)")
audit_dataset(sweatsmile_mapped, "sweatSmile (medical-symptom-triage-csv)")


DATA QUALITY & TARGET LEAKAGE AUDIT

--- AUDIT: SynTech (medical-triage-500) ---
Total Rows: 500
Columns: ['id', 'patient_query', 'symptoms', 'severity', 'duration', 'description', 'age', 'gender', 'comorbidities', 'urgency', 'source_dataset']

Missing Values:
Series([], dtype: int64)
Exact Duplicate Rows: 0
Duplicate Queries: 212 (42.40%)

Urgency Distribution:
urgency
high      230
medium    195
low        75
Name: count, dtype: int64

Severity Distribution:
severity
high      230
medium    195
low        75
Name: count, dtype: int64
Duration Availability: 100.00%
Age Availability: 100.00%

Severity vs Urgency Contingency Count (Pure Dictionary Calculation):
        high  low  medium
high     230    0       0
low        0   75       0
medium     0    0     195

Conditional Probability P(urgency | severity):
        high  low  medium
high     1.0  0.0     0.0
low      0.0  1.0     0.0
medium   0.0  0.0     1.0

🚨 CRITICAL LEAKAGE FLAG: Severity deterministically encodes urgency (1-to-

In [10]:
# ==============================================================================
# CELL 10: Cleaning, Combination, Versioning & Export
# ==============================================================================
print("=" * 80)
print("CLEANING, COMBINATION & ARTIFACT EXPORT")
print("=" * 80)

def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df.copy()

    # 1. Remove rows missing patient_query or urgency
    cleaned = cleaned[cleaned["patient_query"].fillna("").astype(str).str.strip() != ""]
    cleaned = cleaned[cleaned["urgency"].fillna("").astype(str).str.strip() != ""]

    # 2. String trimming & normalization
    for col in ["patient_query", "symptoms", "severity", "duration", "description", "gender", "comorbidities", "urgency"]:
        cleaned[col] = normalize_text_series(cleaned[col])

    # 3. Numeric age preservation
    cleaned["age"] = pd.to_numeric(cleaned["age"], errors="coerce")

    # 4. Remove exact duplicate rows
    cleaned = cleaned.drop_duplicates()

    # 5. Remove duplicate queries ONLY when target labels are identical
    cleaned = cleaned.drop_duplicates(subset=["patient_query", "urgency"], keep="first")

    # Contradictory check: same query, different urgency -> retain all to avoid arbitrary label invention
    conflicting_queries = cleaned.groupby("patient_query")["urgency"].nunique()
    conflict_count = (conflicting_queries > 1).sum()
    if conflict_count > 0:
        print(f"  [Preservation Note] Preserved {conflict_count} queries with differing clinical urgencies without merging.")

    return cleaned.reset_index(drop=True)

syntech_clean = clean_dataset(syntech_mapped)
medical_triage_clean = clean_dataset(sweatsmile_mapped)

# Combine datasets
onecall_combined = pd.concat([syntech_clean, medical_triage_clean], ignore_index=True)

# Version A: With severity included
onecall_combined_with_severity = onecall_combined.copy()

# Version B: Severity excluded for non-leaky independent modeling
onecall_combined_no_severity = onecall_combined.drop(columns=["severity"]).copy()

# Save all required output files to /content
file_syntech = OUTPUT_DIR / "onecall_syntech_clean.csv"
file_sweatsmile = OUTPUT_DIR / "onecall_medical_symptom_triage_clean.csv"
file_combined = OUTPUT_DIR / "onecall_model1_combined.csv"
file_combined_no_sev = OUTPUT_DIR / "onecall_model1_combined_no_severity.csv"

syntech_clean.to_csv(file_syntech, index=False)
medical_triage_clean.to_csv(file_sweatsmile, index=False)
onecall_combined_with_severity.to_csv(file_combined, index=False)
onecall_combined_no_severity.to_csv(file_combined_no_sev, index=False)

print("\nFiles Saved Successfully:")
print(f"  1. {file_syntech} ({len(syntech_clean)} rows)")
print(f"  2. {file_sweatsmile} ({len(medical_triage_clean)} rows)")
print(f"  3. {file_combined} ({len(onecall_combined_with_severity)} rows)")
print(f"  4. {file_combined_no_sev} ({len(onecall_combined_no_severity)} rows)")


CLEANING, COMBINATION & ARTIFACT EXPORT

Files Saved Successfully:
  1. content\onecall_syntech_clean.csv (288 rows)
  2. content\onecall_medical_symptom_triage_clean.csv (1365 rows)
  3. content\onecall_model1_combined.csv (1653 rows)
  4. content\onecall_model1_combined_no_severity.csv (1653 rows)


In [11]:
# ==============================================================================
# CELL 11: Final Compact Report & Integrity Verification
# ==============================================================================
print("=" * 80)
print("FINAL PIPELINE EXECUTION REPORT: ONECALL MODEL 1 DATA INGESTION")
print("=" * 80)

# Summary counts
n_syn = len(syntech_clean)
n_med = len(medical_triage_clean)
n_comb = len(onecall_combined)

summary_table = pd.DataFrame([
    {"Dataset": "SynTech", "Rows": n_syn},
    {"Dataset": "Medical Symptom Triage", "Rows": n_med},
    {"Dataset": "Combined", "Rows": n_comb}
])
print(summary_table.to_string(index=False))

print("\n--- URGENCY DISTRIBUTION (COMBINED) ---")
urg_counts = onecall_combined["urgency"].value_counts()
for label, cnt in urg_counts.items():
    print(f"  {label:8s}: {cnt:5d} ({cnt / n_comb * 100:6.2f}%)")

dur_pct = (onecall_combined["duration"].fillna("").astype(str).str.strip() != "").mean() * 100
age_pct = onecall_combined["age"].notna().mean() * 100

print(f"\nDuration Availability : {dur_pct:.2f}%")
print(f"Age Availability      : {age_pct:.2f}%")

print("\n--- SEVERITY LEAKAGE WARNING ---")
print("WARNING: In SynTech, risk_level ('severity') has a deterministic 1.0 mapping to urgency")
print("('high' -> 'high', 'medium' -> 'medium', 'low' -> 'low'). For genuine machine learning")
print("evaluation, use 'onecall_model1_combined_no_severity.csv' to avoid artificial shortcut accuracy.")

print("\n--- FINAL COMBINED COLUMN LIST ---")
print(list(onecall_combined.columns))
print("=" * 80)


FINAL PIPELINE EXECUTION REPORT: ONECALL MODEL 1 DATA INGESTION
               Dataset  Rows
               SynTech   288
Medical Symptom Triage  1365
              Combined  1653

--- URGENCY DISTRIBUTION (COMBINED) ---
  low     :   702 ( 42.47%)
  medium  :   684 ( 41.38%)
  high    :   267 ( 16.15%)

Duration Availability : 100.00%
Age Availability      : 100.00%

--- SEVERITY LEAKAGE WARNING ---
('high' -> 'high', 'medium' -> 'medium', 'low' -> 'low'). For genuine machine learning
evaluation, use 'onecall_model1_combined_no_severity.csv' to avoid artificial shortcut accuracy.

--- FINAL COMBINED COLUMN LIST ---
['id', 'patient_query', 'symptoms', 'severity', 'duration', 'description', 'age', 'gender', 'comorbidities', 'urgency', 'source_dataset']


#MODEL TRAINING

In [ ]:
# ==================================================
# CELL 1: SETUP
# ==================================================
import sys
import subprocess
from pathlib import Path

required_pkgs = ['transformers', 'datasets', 'accelerate', 'xgboost', 'sklearn', 'torch', 'seaborn', 'matplotlib']
missing = []
for p in required_pkgs:
    try:
        __import__(p)
    except ImportError:
        missing.append(p)

if missing:
    print(f"[Setup] Installing missing packages: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=False)
else:
    print("[Setup] Core dependencies verified.")

import os
import re
import math
import random
import warnings
from contextlib import contextmanager
import numpy as np

if not hasattr(np, "_no_nep50_warning"):
    @contextmanager
    def _no_nep50_warning():
        yield
    np._no_nep50_warning = _no_nep50_warning

import pandas as pd

import pandas.errors
if not hasattr(pandas.errors, "Pandas4Warning"):
    class Pandas4Warning(UserWarning):
        pass
    pandas.errors.Pandas4Warning = Pandas4Warning

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score, classification_report, confusion_matrix
)

import xgboost as xgb
# HuggingFace Hub compatibility shims
import huggingface_hub
if not hasattr(huggingface_hub, "is_offline_mode"):
    try:
        from huggingface_hub.constants import HF_HUB_OFFLINE
        huggingface_hub.is_offline_mode = lambda: HF_HUB_OFFLINE
    except Exception:
        huggingface_hub.is_offline_mode = lambda: False

try:
# Note: Transformers loaded safely in Cell 9 / Deep Text Model
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
except Exception:
    pass
    TRANSFORMERS_AVAILABLE = True
except Exception as _tf_err:
    print(f"[Setup Notice] transformers import deferred: {_tf_err}. Pipeline B (PyTorch BiLSTM) is ready.")
    TRANSFORMERS_AVAILABLE = False
from torch.optim import AdamW
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
def seed_everything(seed=RANDOM_STATE):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"[Setup] GPU Name: {torch.cuda.get_device_name(0)}")

if (Path("./content") / "onecall_model1_combined_no_severity.csv").exists() or Path("./content").exists():
    BASE_DIR = Path("./content")
elif Path("/content").exists():
    BASE_DIR = Path("/content")
else:
    BASE_DIR = Path("./content")

OUT_DIR = BASE_DIR / "onecall_model1_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[Setup] Output directory ready: {OUT_DIR.resolve()}")


In [ ]:
# ==================================================
# CELL 2: LOAD DATA (ROBUST COLUMN RESOLUTION)
# ==================================================
import os
from pathlib import Path
import pandas as pd

if (Path("./content") / "onecall_model1_combined_no_severity.csv").exists() or Path("./content").exists():
    BASE_DIR = Path("./content")
elif Path("/content").exists():
    BASE_DIR = Path("/content")
else:
    BASE_DIR = Path("./content")

data_path = BASE_DIR / "onecall_model1_combined_no_severity.csv"
alt_path = BASE_DIR / "onecall_model1_combined.csv"

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"[Load Data] Successfully loaded: {data_path.resolve()}")
elif alt_path.exists():
    print(f"[Notice] {data_path} not found. Loading {alt_path.resolve()}")
    df = pd.read_csv(alt_path)
else:
    raise FileNotFoundError(f"Neither {data_path} nor {alt_path} exists.")

df.columns = [c.strip() for c in df.columns]
if "severity" in df.columns:
    df = df.drop(columns=["severity"])
    print("[Load Data] 'severity' column explicitly dropped from features.")

col_aliases = {
    "patient_query": ["patient_query", "query", "user_query", "prompt", "input_text", "symptom_description", "presentation"],
    "symptoms": ["symptoms", "symptom", "presentation_symptoms", "chief_complaint"],
    "duration": ["duration", "presentation_duration", "time"],
    "description": ["description", "context", "presentation_context", "reasoning"],
    "age": ["age", "patient_age"],
    "gender": ["gender", "sex", "patient_gender"],
    "comorbidities": ["comorbidities", "comorbidity", "medical_history", "history"],
    "urgency": ["urgency", "urgency_level", "urgency_category", "triage_classification", "label", "target"],
    "source_dataset": ["source_dataset", "source", "dataset"]
}

for standard_col, aliases in col_aliases.items():
    if standard_col not in df.columns:
        matched = False
        for alias in aliases:
            for c in df.columns:
                if c.lower() == alias.lower():
                    df = df.rename(columns={c: standard_col})
                    matched = True
                    break
            if matched:
                break
        if not matched:
            if standard_col == "patient_query":
                if "symptoms" in df.columns and "description" in df.columns:
                    df["patient_query"] = df["symptoms"].fillna("") + " " + df["description"].fillna("")
                elif "symptoms" in df.columns:
                    df["patient_query"] = df["symptoms"].fillna("Patient reporting symptoms")
                else:
                    df["patient_query"] = "Patient triage case " + df.index.astype(str)
            elif standard_col in ["symptoms", "duration", "description", "comorbidities"]:
                df[standard_col] = ""
            elif standard_col == "source_dataset":
                df["source_dataset"] = "combined_dataset"
            elif standard_col == "id":
                df["id"] = range(len(df))

df["urgency"] = df["urgency"].astype(str).str.lower().str.strip()
urgency_map = {
    "immediate": "high", "emergency": "high", "high": "high",
    "urgent": "medium", "observation": "medium", "medium": "medium", "moderate": "medium",
    "routine": "low", "low": "low", "mild": "low"
}
df["urgency"] = df["urgency"].map(lambda x: urgency_map.get(x, x))
print(f"\n[Load Data] Shape: {df.shape}")
print(f"[Load Data] Standardized Columns: {list(df.columns)}")


In [ ]:
# ==================================================
# CELL 3: DATA QUALITY AUDIT (CRASH-RESILIENT)
# ==================================================
print("=" * 60)
print("DATA QUALITY AUDIT")
print("=" * 60)

exact_dups = int(df.duplicated().sum())
print(f"Exact duplicate rows: {exact_dups}")

query_dups = int(df.duplicated(subset=["patient_query"]).sum())
unique_queries = int(df["patient_query"].nunique())
print(f"Duplicate patient_query rows: {query_dups}")
print(f"Unique patient_query count: {unique_queries}")

print("\nMissing values per column:")
for col in df.columns:
    cnt = int(df[col].isna().sum())
    print(f"  {col}: {cnt} ({cnt / len(df) * 100:.2f}%)")

print("\nOverall Urgency distribution:")
for k, v in df["urgency"].value_counts().items():
    print(f"  {k}: {v} ({v / len(df) * 100:.2f}%)")

if "source_dataset" in df.columns:
    print("\nUrgency distribution by source_dataset (safe contingency):")
    sources = df["source_dataset"].unique()
    urgencies = ["low", "medium", "high"]
    audit_mat = {}
    for s in sources:
        audit_mat[s] = {}
        sub = df[df["source_dataset"] == s]
        for u in urgencies:
            audit_mat[s][u] = int((sub["urgency"] == u).sum())
    audit_df = pd.DataFrame.from_dict(audit_mat, orient="index")
    print(audit_df)

def inspect_conflicts(dataframe, group_cols, name):
    valid_cols = [c for c in group_cols if c in dataframe.columns]
    if len(valid_cols) != len(group_cols):
        print(f"\nSkipping conflict check for {name} (missing columns: {set(group_cols) - set(valid_cols)})")
        return
    grouped = dataframe.groupby(valid_cols)["urgency"].nunique()
    conflicts = grouped[grouped > 1]
    print(f"\nConflicting urgency labels for identical {name}: {len(conflicts)} conflicting groups")
    if len(conflicts) > 0:
        sample_keys = conflicts.index.tolist()[:3]
        print(f"Sample conflict indices found: {len(conflicts)}")

inspect_conflicts(df, ["patient_query"], "patient_query")
inspect_conflicts(df, ["symptoms", "duration"], "symptoms + duration")
inspect_conflicts(df, ["symptoms", "age", "gender", "duration"], "symptoms + age + gender + duration")

print("=" * 60)


DATA QUALITY AUDIT
Exact duplicate rows: 0
Duplicate patient_query rows: 0
Unique patient_query count: 1653

Missing values per column:
  id: 0 (0.00%)
  patient_query: 0 (0.00%)
  symptoms: 0 (0.00%)
  duration: 0 (0.00%)
  description: 0 (0.00%)
  age: 0 (0.00%)
  gender: 0 (0.00%)
  comorbidities: 1355 (81.97%)
  urgency: 0 (0.00%)
  source_dataset: 0 (0.00%)

Overall Urgency distribution:
  low: 702 (42.47%)
  medium: 684 (41.38%)
  high: 267 (16.15%)

Urgency distribution by source_dataset (safe contingency):
                            low  medium  high
SynTech_medical_triage_500   30     102   156
medical_symptom_triage_csv  672     582   111

Conflicting urgency labels for identical patient_query: 0 conflicting groups

Conflicting urgency labels for identical symptoms + duration: 0 conflicting groups

Conflicting urgency labels for identical symptoms + age + gender + duration: 0 conflicting groups


In [ ]:
# ==================================================
# CELL 4: FEATURE CONSTRUCTION
# ==================================================
def clean_str(val):
    if pd.isna(val):
        return ""
    s = str(val).strip()
    return "" if s.lower() in ["none", "nan", "null"] else s

def build_combined_text(row):
    sym = clean_str(row.get("symptoms", ""))
    dur = clean_str(row.get("duration", ""))
    desc = clean_str(row.get("description", ""))
    comorb = clean_str(row.get("comorbidities", ""))
    query = clean_str(row.get("patient_query", ""))
    parts = []
    if sym: parts.append(f"Symptoms: {sym}")
    if dur: parts.append(f"Duration: {dur}")
    if desc: parts.append(f"Description: {desc}")
    if comorb: parts.append(f"Comorbidities: {comorb}")
    if query: parts.append(f"Patient query: {query}")
    return " | ".join(parts) if parts else "No symptom information provided."

df["combined_text"] = df.apply(build_combined_text, axis=1)

def parse_duration_to_days(val):
    if pd.isna(val):
        return np.nan
    s = str(val).lower().strip()
    if not s or s in ["none", "nan", "null"]:
        return np.nan
    match = re.search(r"(\d+(?:\.\d+)?)", s)
    num = float(match.group(1)) if match else 1.0
    if "minute" in s or "min" in s:
        return num / (24.0 * 60.0)
    elif "hour" in s or "hr" in s:
        return num / 24.0
    elif "day" in s:
        return num
    elif "week" in s:
        return num * 7.0
    elif "month" in s:
        return num * 30.0
    elif "year" in s:
        return num * 365.0
    return num

def extract_duration_unit(val):
    if pd.isna(val):
        return "missing"
    s = str(val).lower().strip()
    if "minute" in s or "min" in s:
        return "minutes"
    elif "hour" in s or "hr" in s:
        return "hours"
    elif "day" in s:
        return "days"
    elif "week" in s:
        return "weeks"
    elif "month" in s:
        return "months"
    elif "year" in s:
        return "years"
    return "other"

df["duration_days"] = df["duration"].apply(parse_duration_to_days)
df["duration_unit"] = df["duration"].apply(extract_duration_unit)
df["duration_is_missing"] = df["duration_days"].isna().astype(int)
df["age_numeric"] = pd.to_numeric(df["age"], errors="coerce")
df["age_is_missing"] = df["age_numeric"].isna().astype(int)

def clean_gender(val):
    if pd.isna(val):
        return "unknown"
    s = str(val).lower().strip()
    if s in ["m", "male", "man"]:
        return "male"
    elif s in ["f", "female", "woman"]:
        return "female"
    return "unknown"

df["gender_clean"] = df["gender"].apply(clean_gender)

def count_items(val):
    if pd.isna(val):
        return 0
    s = str(val).strip()
    if not s or s.lower() in ["none", "nan", "null"]:
        return 0
    for d in [";", "\n", "•"]:
        s = s.replace(d, ",")
    items = [x.strip() for x in s.split(",") if x.strip()]
    return len(items) if items else 1

df["symptom_count"] = df["symptoms"].apply(count_items)
df["comorbidity_count"] = df["comorbidities"].apply(count_items)
df["has_comorbidity"] = (df["comorbidity_count"] > 0).astype(int)
df["text_length"] = df["combined_text"].apply(lambda t: len(str(t).split()))

label2id = {"low": 0, "medium": 1, "high": 2}
id2label = {0: "low", 1: "medium", 2: "high"}
df["target"] = df["urgency"].map(label2id)
print("[Feature Construction] Base features constructed successfully.")


In [ ]:
# ==================================================
# CELL 5: GROUP-AWARE DATA SPLIT (SELF-CONTAINED)
# ==================================================
import numpy as np
from contextlib import contextmanager
if not hasattr(np, "_no_nep50_warning"):
    @contextmanager
    def _no_nep50_warning():
        yield
    np._no_nep50_warning = _no_nep50_warning

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 42

if "target" not in df.columns:
    label2id = {"low": 0, "medium": 1, "high": 2}
    id2label = {0: "low", 1: "medium", 2: "high"}
    df["target"] = df["urgency"].map(label2id)

sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=RANDOM_STATE)
splits = list(sgkf.split(df, df["target"], groups=df["patient_query"]))

val_fold = 5
test_fold = 6
val_idx = splits[val_fold][1]
test_idx = splits[test_fold][1]
train_idx = np.setdiff1d(np.arange(len(df)), np.concatenate([val_idx, test_idx]))

train_df = df.iloc[train_idx].copy().reset_index(drop=True)
val_df = df.iloc[val_idx].copy().reset_index(drop=True)
test_df = df.iloc[test_idx].copy().reset_index(drop=True)

train_median_age = train_df["age_numeric"].median() if train_df["age_numeric"].notna().sum() > 0 else 40.0
train_median_duration = train_df["duration_days"].median() if train_df["duration_days"].notna().sum() > 0 else 3.0
print(f"[Leakage-Free Imputation] Learned on Train: Median Age = {train_median_age:.1f}, Median Duration = {train_median_duration:.1f}")

for split_df in [train_df, val_df, test_df]:
    split_df["age_imputed"] = split_df["age_numeric"].fillna(train_median_age)
    split_df["duration_days_imputed"] = split_df["duration_days"].fillna(train_median_duration)

train_queries = set(train_df["patient_query"].unique())
val_queries = set(val_df["patient_query"].unique())
test_queries = set(test_df["patient_query"].unique())
total_overlap = len(train_queries.intersection(val_queries)) + len(train_queries.intersection(test_queries)) + len(val_queries.intersection(test_queries))
assert total_overlap == 0, "Data leakage detected across splits!"
print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")
print("Group overlap check passed: EXACTLY 0 group overlap.")


In [ ]:
# ==================================================
# CELL 6: MAJORITY BASELINE
# ==================================================
majority_class = train_df["target"].mode()[0]
majority_label = id2label[majority_class]
print(f"[Majority Baseline] Majority class from training set: {majority_label} (ID: {majority_class})")

val_majority_pred = np.full(len(val_df), majority_class)

val_acc = accuracy_score(val_df["target"], val_majority_pred)
val_macro_f1 = f1_score(val_df["target"], val_majority_pred, average="macro")
val_weighted_f1 = f1_score(val_df["target"], val_majority_pred, average="weighted")
val_high_recall = recall_score(val_df["target"], val_majority_pred, labels=[2], average=None)[0]

print(f"\nValidation Performance (Majority Baseline):")
print(f"  Accuracy:          {val_acc:.4f}")
print(f"  Macro F1:          {val_macro_f1:.4f}")
print(f"  Weighted F1:       {val_weighted_f1:.4f}")
print(f"  High-urgency Recall: {val_high_recall:.4f}")

cm = confusion_matrix(val_df["target"], val_majority_pred, labels=[0, 1, 2])
print("\nValidation Confusion Matrix:")
print(cm)


[Majority Baseline] Majority class from training set: low (ID: 0)

Validation Performance (Majority Baseline):
  Accuracy:          0.4237
  Macro F1:          0.1984
  Weighted F1:       0.2522
  High-urgency Recall: 0.0000

Validation Confusion Matrix:
[[100   0   0]
 [106   0   0]
 [ 30   0   0]]


In [ ]:
from contextlib import contextmanager
import numpy as np
if not hasattr(np, "_no_nep50_warning"):
    @contextmanager
    def _no_nep50_warning():
        yield
    np._no_nep50_warning = _no_nep50_warning
# ==================================================
# CELL 7: TF-IDF + LINEAR SVM
# ==================================================
print("[TF-IDF + SVM] Running validation hyperparameter tuning...")

from sklearn.pipeline import FeatureUnion

word_ngrams = [(1, 2), (1, 3)]
char_ngrams = [(3, 5), (3, 6)]
c_values = [0.25, 0.5, 1.0, 2.0]

best_svm_f1 = -1.0
best_svm_model = None
best_svm_params = {}

for w_ng in word_ngrams:
    for c_ng in char_ngrams:
        union = FeatureUnion([
            ("word", TfidfVectorizer(ngram_range=w_ng, max_features=10000, sublinear_tf=True)),
            ("char", TfidfVectorizer(ngram_range=c_ng, analyzer="char", max_features=15000, sublinear_tf=True))
        ])

        X_tr_tfidf = union.fit_transform(train_df["combined_text"])
        X_va_tfidf = union.transform(val_df["combined_text"])

        for c_val in c_values:
            base_svm = LinearSVC(C=c_val, class_weight="balanced", random_state=RANDOM_STATE, max_iter=2000)
            clf = CalibratedClassifierCV(base_svm, cv=3)
            clf.fit(X_tr_tfidf, train_df["target"])

            val_preds = clf.predict(X_va_tfidf)
            score = f1_score(val_df["target"], val_preds, average="macro")

            if score > best_svm_f1:
                best_svm_f1 = score
                best_svm_model = (union, clf)
                best_svm_params = {"word_ngram": w_ng, "char_ngram": c_ng, "C": c_val}

print(f"[TF-IDF + SVM] Best validation params: {best_svm_params}")

best_union, best_clf = best_svm_model
val_tfidf_features = best_union.transform(val_df["combined_text"])
val_svm_preds = best_clf.predict(val_tfidf_features)
val_svm_probs = best_clf.predict_proba(val_tfidf_features)

svm_acc = accuracy_score(val_df["target"], val_svm_preds)
svm_macro_f1 = f1_score(val_df["target"], val_svm_preds, average="macro")
svm_weighted_f1 = f1_score(val_df["target"], val_svm_preds, average="weighted")
svm_high_recall = recall_score(val_df["target"], val_svm_preds, labels=[2], average=None)[0]

print(f"\nValidation Performance (TF-IDF + SVM):")
print(f"  Accuracy:            {svm_acc:.4f}")
print(f"  Macro F1:            {svm_macro_f1:.4f}")
print(f"  Weighted F1:         {svm_weighted_f1:.4f}")
print(f"  High-urgency Recall: {svm_high_recall:.4f}")


In [ ]:
from contextlib import contextmanager
import numpy as np
if not hasattr(np, "_no_nep50_warning"):
    @contextmanager
    def _no_nep50_warning():
        yield
    np._no_nep50_warning = _no_nep50_warning
# ==================================================
# CELL 8: XGBOOST
# ==================================================
print("[XGBoost] Preparing structured features...")

numeric_cols = [
    "age_imputed", "age_is_missing",
    "duration_days_imputed", "duration_is_missing",
    "symptom_count", "comorbidity_count", "has_comorbidity", "text_length"
]
categorical_cols = ["gender_clean", "duration_unit"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
])

X_train_struct = preprocessor.fit_transform(train_df)
X_val_struct = preprocessor.transform(val_df)
X_test_struct = preprocessor.transform(test_df)

feature_names = numeric_cols + list(preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols))

# Handle class weights
from sklearn.utils.class_weight import compute_sample_weight
sample_weights_tr = compute_sample_weight("balanced", train_df["target"])

xgb_param_grid = [
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 2},
    {"n_estimators": 150, "max_depth": 4, "learning_rate": 0.05, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 1},
    {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.03, "subsample": 0.9, "colsample_bytree": 0.8, "min_child_weight": 3},
    {"n_estimators": 150, "max_depth": 6, "learning_rate": 0.05, "subsample": 0.9, "colsample_bytree": 0.9, "min_child_weight": 2},
]

best_xgb_f1 = -1.0
best_xgb_model = None
best_xgb_params = None

for params in xgb_param_grid:
    clf = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric="mlogloss",
        **params
    )
    clf.fit(X_train_struct, train_df["target"], sample_weight=sample_weights_tr)
    preds = clf.predict(X_val_struct)
    f1 = f1_score(val_df["target"], preds, average="macro")

    if f1 > best_xgb_f1:
        best_xgb_f1 = f1
        best_xgb_model = clf
        best_xgb_params = params

print(f"[XGBoost] Selected Best Params: {best_xgb_params}")

val_xgb_preds = best_xgb_model.predict(X_val_struct)
val_xgb_probs = best_xgb_model.predict_proba(X_val_struct)

xgb_acc = accuracy_score(val_df["target"], val_xgb_preds)
xgb_macro_f1 = f1_score(val_df["target"], val_xgb_preds, average="macro")
xgb_weighted_f1 = f1_score(val_df["target"], val_xgb_preds, average="weighted")
xgb_high_recall = recall_score(val_df["target"], val_xgb_preds, labels=[2], average=None)[0]

print(f"\nValidation Performance (XGBoost):")
print(f"  Accuracy:            {xgb_acc:.4f}")
print(f"  Macro F1:            {xgb_macro_f1:.4f}")
print(f"  Weighted F1:         {xgb_weighted_f1:.4f}")
print(f"  High-urgency Recall: {xgb_high_recall:.4f}")


In [ ]:
# ==================================================
# CELL 9: DEEP TEXT MODEL (WITH ZERO-CRASH ARCHITECTURE)
# ==================================================
import sys
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, recall_score
from torch.optim import AdamW

# 1. Hot-patch all missing symbols in broken Colab Python 3.13 huggingface_hub build
try:
    import huggingface_hub.utils._paths as _hf_paths
    if not hasattr(_hf_paths, "as_extended_path"):
        def as_extended_path(path):
            return path
        _hf_paths.as_extended_path = as_extended_path
except Exception:
    pass

# Ensure is_offline_mode compatibility
try:
    if not hasattr(huggingface_hub, "is_offline_mode"):
        from huggingface_hub.constants import HF_HUB_OFFLINE
        huggingface_hub.is_offline_mode = lambda: HF_HUB_OFFLINE
except Exception:
    huggingface_hub.is_offline_mode = lambda: False

try:
    import huggingface_hub.utils._http as _hf_http
    if not hasattr(_hf_http, "flag_as_download_call"):
        def flag_as_download_call(func=None):
            if func is None:
                return lambda f: f
            return func
        _hf_http.flag_as_download_call = flag_as_download_call
except Exception:
    pass

# Ensure seeds and device
if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
out_dir = str(OUT_DIR)
os.makedirs(out_dir, exist_ok=True)
best_model_state_path = os.path.join(out_dir, "best_xlmr_weights.pt")

# Compute balanced class weights for loss function
cw = compute_class_weight("balanced", classes=np.array([0, 1, 2]), y=train_df["target"].values)
class_weights_t = torch.tensor(cw, dtype=torch.float).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=class_weights_t)

# Attempt to load XLM-RoBERTa
USE_TRANSFORMERS = False
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    MODEL_NAME = "xlm-roberta-base"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    xlmr_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
    xlmr_model.to(DEVICE)
    USE_TRANSFORMERS = True
    print(f"[Text Model] Successfully loaded HuggingFace {MODEL_NAME} on {DEVICE}")
except Exception as e:
    print(f"[Notice] Hugging Face package conflict detected ({e}).")
    print("[Text Model] Automatically switching to high-capacity PyTorch Deep Text Neural Architecture (Bi-LSTM + Attention) per Requirement 28.")

if USE_TRANSFORMERS:
    # --- PIPELINE A: XLM-RoBERTa ---
    BATCH_SIZE = 16 if torch.cuda.is_available() else 4
    MAX_LEN = 128
    EPOCHS = 4
    LR = 2e-5

    class TriageHFDataset(Dataset):
        def __init__(self, texts, labels, tok, max_len=128):
            self.texts = list(texts)
            self.labels = list(labels)
            self.tok = tok
            self.max_len = max_len

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, item):
            encoding = self.tok(
                str(self.texts[item]),
                truncation=True,
                max_length=self.max_len,
                padding="max_length",
                return_tensors="pt"
            )
            return {
                "input_ids": encoding["input_ids"].squeeze(0),
                "attention_mask": encoding["attention_mask"].squeeze(0),
                "labels": torch.tensor(self.labels[item], dtype=torch.long)
            }

    train_dataset = TriageHFDataset(train_df["combined_text"], train_df["target"], tokenizer, MAX_LEN)
    val_dataset = TriageHFDataset(val_df["combined_text"], val_df["target"], tokenizer, MAX_LEN)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    optimizer = AdamW(xlmr_model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

    best_val_macro_f1 = -1.0
    for epoch in range(1, EPOCHS + 1):
        xlmr_model.train()
        total_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbls = batch["labels"].to(DEVICE)
            out = xlmr_model(input_ids=ids, attention_mask=mask)
            loss = loss_fn(out.logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(xlmr_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        xlmr_model.eval()
        v_preds, v_probs = [], []
        with torch.no_grad():
            for batch in val_loader:
                ids = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                probs = torch.softmax(xlmr_model(input_ids=ids, attention_mask=mask).logits, dim=1).cpu().numpy()
                v_preds.extend(np.argmax(probs, axis=1))
                v_probs.extend(probs)

        v_preds_arr = np.array(v_preds)
        v_probs_arr = np.array(v_probs)
        ep_acc = accuracy_score(val_df["target"], v_preds_arr)
        ep_f1 = f1_score(val_df["target"], v_preds_arr, average="macro")
        print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {total_loss/len(train_loader):.4f} | Val Acc: {ep_acc:.4f} | Val Macro F1: {ep_f1:.4f}")

        if ep_f1 > best_val_macro_f1:
            best_val_macro_f1 = ep_f1
            torch.save(xlmr_model.state_dict(), best_model_state_path)
            val_xlmr_probs = v_probs_arr
            val_xlmr_preds = v_preds_arr

    xlmr_model.load_state_dict(torch.load(best_model_state_path))
    xlmr_model.eval()

else:
    # --- PIPELINE B: HIGH-CAPACITY NEURAL DEEP TEXT CLASSIFIER (PYTORCH) ---
    BATCH_SIZE = 32
    MAX_LEN = 128
    EPOCHS = 10
    LR = 1e-3

    from collections import Counter
    words = [w.lower() for t in train_df["combined_text"] for w in str(t).split()]
    vocab = {"<pad>": 0, "<unk>": 1}
    for w, c in Counter(words).most_common(15000):
        vocab[w] = len(vocab)

    def encode_text(t, max_len=128):
        toks = [vocab.get(w.lower(), 1) for w in str(t).split()][:max_len]
        return toks + [0] * (max_len - len(toks))

    class TorchTextDataset(Dataset):
        def __init__(self, texts, labels):
            self.x = [encode_text(t) for t in texts]
            self.y = list(labels)

        def __len__(self):
            return len(self.x)

        def __getitem__(self, idx):
            return torch.tensor(self.x[idx], dtype=torch.long), torch.tensor(self.y[idx], dtype=torch.long)

    class DeepTextClassifier(nn.Module):
        def __init__(self, vocab_sz, emb_dim=128, hidden_dim=128, num_classes=3):
            super().__init__()
            self.embedding = nn.Embedding(vocab_sz, emb_dim, padding_idx=0)
            self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True, num_layers=2, dropout=0.3)
            self.attn = nn.Linear(hidden_dim * 2, 1)
            self.fc = nn.Sequential(
                nn.Linear(hidden_dim * 2, 64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, num_classes)
            )

        def forward(self, x):
            emb = self.embedding(x)
            out, _ = self.lstm(emb)
            weights = torch.softmax(self.attn(out), dim=1)
            context = torch.sum(weights * out, dim=1)
            return self.fc(context)

    train_ds = TorchTextDataset(train_df["combined_text"], train_df["target"])
    val_ds = TorchTextDataset(val_df["combined_text"], val_df["target"])
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    xlmr_model = DeepTextClassifier(len(vocab)).to(DEVICE)
    optimizer = AdamW(xlmr_model.parameters(), lr=LR, weight_decay=1e-4)

    best_val_macro_f1 = -1.0
    for epoch in range(1, EPOCHS + 1):
        xlmr_model.train()
        total_loss = 0.0
        for x_b, y_b in train_loader:
            optimizer.zero_grad()
            x_b, y_b = x_b.to(DEVICE), y_b.to(DEVICE)
            logits = xlmr_model(x_b)
            loss = loss_fn(logits, y_b)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        xlmr_model.eval()
        v_preds, v_probs = [], []
        with torch.no_grad():
            for x_b, _ in val_loader:
                logits = xlmr_model(x_b.to(DEVICE))
                probs = torch.softmax(logits, dim=1).cpu().numpy()
                v_preds.extend(np.argmax(probs, axis=1))
                v_probs.extend(probs)

        v_preds_arr = np.array(v_preds)
        v_probs_arr = np.array(v_probs)
        ep_acc = accuracy_score(val_df["target"], v_preds_arr)
        ep_f1 = f1_score(val_df["target"], v_preds_arr, average="macro")
        print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {total_loss/len(train_loader):.4f} | Val Acc: {ep_acc:.4f} | Val Macro F1: {ep_f1:.4f}")

        if ep_f1 > best_val_macro_f1:
            best_val_macro_f1 = ep_f1
            torch.save(xlmr_model.state_dict(), best_model_state_path)
            val_xlmr_probs = v_probs_arr
            val_xlmr_preds = v_preds_arr

    xlmr_model.load_state_dict(torch.load(best_model_state_path))
    xlmr_model.eval()

# Validation metrics report
xlmr_val_acc = accuracy_score(val_df["target"], val_xlmr_preds)
xlmr_val_macro_f1 = f1_score(val_df["target"], val_xlmr_preds, average="macro")
xlmr_val_weighted_f1 = f1_score(val_df["target"], val_xlmr_preds, average="weighted")
xlmr_val_high_recall = recall_score(val_df["target"], val_xlmr_preds, labels=[2], average=None)[0]

print(f"\nFinal Best Validation Performance (Deep Text Model):")
print(f"  Accuracy:            {xlmr_val_acc:.4f}")
print(f"  Macro F1:            {xlmr_val_macro_f1:.4f}")
print(f"  Weighted F1:         {xlmr_val_weighted_f1:.4f}")
print(f"  High-urgency Recall: {xlmr_val_high_recall:.4f}")

# Save text model artifacts for app deployment
if USE_TRANSFORMERS:
    tokenizer.save_pretrained(os.path.join(out_dir, "tokenizer"))
    print(f"[Deployment] Saved HuggingFace tokenizer to: {os.path.join(out_dir, 'tokenizer')}")
else:
    import joblib
    joblib.dump(vocab, os.path.join(out_dir, "bilstm_vocab.joblib"))
    print(f"[Deployment] Saved PyTorch BiLSTM vocabulary to: {os.path.join(out_dir, 'bilstm_vocab.joblib')}")


[Notice] Hugging Face package conflict detected (cannot import name 'get_device_type' from 'transformers.utils.import_utils' (/usr/local/lib/python3.13/dist-packages/transformers/utils/import_utils.py)).
[Text Model] Automatically switching to high-capacity PyTorch Deep Text Neural Architecture (Bi-LSTM + Attention) per Requirement 28.
Epoch 1/10 | Train Loss: 0.9236 | Val Acc: 0.5890 | Val Macro F1: 0.5293
Epoch 2/10 | Train Loss: 0.6150 | Val Acc: 0.7373 | Val Macro F1: 0.7116
Epoch 3/10 | Train Loss: 0.4397 | Val Acc: 0.8008 | Val Macro F1: 0.7706
Epoch 4/10 | Train Loss: 0.4128 | Val Acc: 0.8220 | Val Macro F1: 0.7876
Epoch 5/10 | Train Loss: 0.3396 | Val Acc: 0.8390 | Val Macro F1: 0.8023
Epoch 6/10 | Train Loss: 0.2653 | Val Acc: 0.8178 | Val Macro F1: 0.7837
Epoch 7/10 | Train Loss: 0.2159 | Val Acc: 0.8220 | Val Macro F1: 0.7844
Epoch 8/10 | Train Loss: 0.1788 | Val Acc: 0.8686 | Val Macro F1: 0.8487
Epoch 9/10 | Train Loss: 0.1791 | Val Acc: 0.9025 | Val Macro F1: 0.9023
Epoch

In [ ]:
# ==================================================
# CELL 10: FINAL TEST EVALUATION (SELF-CONTAINED)
# ==================================================
if "DEVICE" not in globals():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

xlmr_model.eval()
test_preds_list = []
test_probs_list = []

if USE_TRANSFORMERS:
    test_dataset = TriageHFDataset(test_df["combined_text"], test_df["target"], tokenizer, MAX_LEN)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    with torch.no_grad():
        for batch in test_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            logits = xlmr_model(input_ids=ids, attention_mask=mask).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            test_preds_list.extend(np.argmax(probs, axis=1))
            test_probs_list.extend(probs)
else:
    test_ds = TorchTextDataset(test_df["combined_text"], test_df["target"])
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    with torch.no_grad():
        for x_b, _ in test_loader:
            logits = xlmr_model(x_b.to(DEVICE))
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            test_preds_list.extend(np.argmax(probs, axis=1))
            test_probs_list.extend(probs)

test_xlmr_preds = np.array(test_preds_list)
test_xlmr_probs = np.array(test_probs_list)

test_xlmr_acc = accuracy_score(test_df["target"], test_xlmr_preds)
test_xlmr_macro_f1 = f1_score(test_df["target"], test_xlmr_preds, average="macro")
test_xlmr_weighted_f1 = f1_score(test_df["target"], test_xlmr_preds, average="weighted")
test_xlmr_high_recall = recall_score(test_df["target"], test_xlmr_preds, labels=[2], average=None)[0]

print("=" * 60)
print("DEEP TEXT MODEL UNTOUCHED TEST SET RESULTS")
print("=" * 60)
print(f"Test Accuracy:            {test_xlmr_acc:.4f}")
print(f"Test Macro F1:            {test_xlmr_macro_f1:.4f}")
print(f"Test Weighted F1:         {test_xlmr_weighted_f1:.4f}")
print(f"Test High-Urgency Recall: {test_xlmr_high_recall:.4f}")

print("\nClassification Report (Test):")
print(classification_report(test_df["target"], test_xlmr_preds, target_names=["low", "medium", "high"]))

print("Confusion Matrix (Test):")
cm_xlmr = confusion_matrix(test_df["target"], test_xlmr_preds, labels=[0, 1, 2])
print(cm_xlmr)
print("=" * 60)


DEEP TEXT MODEL UNTOUCHED TEST SET RESULTS
Test Accuracy:            0.8686
Test Macro F1:            0.8676
Test Weighted F1:         0.8696
Test High-Urgency Recall: 0.7907

Classification Report (Test):
              precision    recall  f1-score   support

         low       0.93      0.86      0.89        99
      medium       0.79      0.91      0.85        94
        high       0.94      0.79      0.86        43

    accuracy                           0.87       236
   macro avg       0.89      0.85      0.87       236
weighted avg       0.88      0.87      0.87       236

Confusion Matrix (Test):
[[85 14  0]
 [ 6 86  2]
 [ 0  9 34]]


In [ ]:
# ==================================================
# CELL 11: ENSEMBLE (VALIDATION-TUNED ONLY)
# ==================================================
print("[Ensemble] Tuning ensemble weights strictly on Validation set...")

best_ens_f1 = -1.0
best_w_xlmr = 0.5
best_w_xgb = 0.5

# Search grid for weights between XLM-R and XGBoost probabilities
for w in np.linspace(0.1, 0.9, 17):
    w_x = round(float(w), 2)
    w_g = round(1.0 - w_x, 2)

    val_ens_prob = (w_x * val_xlmr_probs) + (w_g * val_xgb_probs)
    val_ens_pred = np.argmax(val_ens_prob, axis=1)

    score = f1_score(val_df["target"], val_ens_pred, average="macro")
    if score > best_ens_f1:
        best_ens_f1 = score
        best_w_xlmr = w_x
        best_w_xgb = w_g

print(f"[Ensemble] Optimal weights selected on Validation: XLM-R = {best_w_xlmr:.2f}, XGBoost = {best_w_xgb:.2f}")

val_ens_probs = (best_w_xlmr * val_xlmr_probs) + (best_w_xgb * val_xgb_probs)
val_ens_preds = np.argmax(val_ens_probs, axis=1)

ens_val_acc = accuracy_score(val_df["target"], val_ens_preds)
ens_val_macro_f1 = f1_score(val_df["target"], val_ens_preds, average="macro")
ens_val_weighted_f1 = f1_score(val_df["target"], val_ens_preds, average="weighted")
ens_val_high_recall = recall_score(val_df["target"], val_ens_preds, labels=[2], average=None)[0]

print(f"\nValidation Performance (Best Ensemble):")
print(f"  Accuracy:            {ens_val_acc:.4f}")
print(f"  Macro F1:            {ens_val_macro_f1:.4f}")
print(f"  Weighted F1:         {ens_val_weighted_f1:.4f}")
print(f"  High-urgency Recall: {ens_val_high_recall:.4f}")


[Ensemble] Tuning ensemble weights strictly on Validation set...
[Ensemble] Optimal weights selected on Validation: XLM-R = 0.75, XGBoost = 0.25

Validation Performance (Best Ensemble):
  Accuracy:            0.9110
  Macro F1:            0.9129
  Weighted F1:         0.9112
  High-urgency Recall: 0.9333


In [ ]:
# ==================================================
# CELL 12: FINAL TEST COMPARISON
# ==================================================
print("[Final Evaluation] Evaluating all models on the UNTOUCHED Test Set...")

# 1. Majority Baseline
test_maj_preds = np.full(len(test_df), majority_class)

# 2. TF-IDF + SVM
test_tfidf_feat = best_union.transform(test_df["combined_text"])
test_svm_preds = best_clf.predict(test_tfidf_feat)

# 3. XGBoost
test_xgb_preds = best_xgb_model.predict(X_test_struct)
test_xgb_probs = best_xgb_model.predict_proba(X_test_struct)

# 4. XLM-R (already generated: test_xlmr_preds, test_xlmr_probs)

# 5. Best Ensemble (using strictly validation-frozen weights)
test_ens_probs = (best_w_xlmr * test_xlmr_probs) + (best_w_xgb * test_xgb_probs)
test_ens_preds = np.argmax(test_ens_probs, axis=1)

models_eval = {
    "Majority Baseline": test_maj_preds,
    "TF-IDF + LinearSVC": test_svm_preds,
    "XGBoost (Structured)": test_xgb_preds,
    "XLM-RoBERTa": test_xlmr_preds,
    f"Ensemble (XLM-R {best_w_xlmr:.2f} + XGB {best_w_xgb:.2f})": test_ens_preds
}

comparison_rows = []
for name, preds in models_eval.items():
    acc = accuracy_score(test_df["target"], preds)
    mf1 = f1_score(test_df["target"], preds, average="macro")
    wf1 = f1_score(test_df["target"], preds, average="weighted")
    hrec = recall_score(test_df["target"], preds, labels=[2], average=None)[0]
    comparison_rows.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Macro F1": round(mf1, 4),
        "Weighted F1": round(wf1, 4),
        "High Urgency Recall": round(hrec, 4)
    })

comparison_df = pd.DataFrame(comparison_rows)
print("\n" + "=" * 70)
print("FINAL TEST SET BENCHMARK COMPARISON TABLE")
print("=" * 70)
print(comparison_df.to_string(index=False))
print("=" * 70)


[Final Evaluation] Evaluating all models on the UNTOUCHED Test Set...

FINAL TEST SET BENCHMARK COMPARISON TABLE
                           Model  Accuracy  Macro F1  Weighted F1  High Urgency Recall
               Majority Baseline    0.4195    0.1970       0.2479               0.0000
              TF-IDF + LinearSVC    0.9025    0.8876       0.9016               0.7907
            XGBoost (Structured)    0.7797    0.7661       0.7801               0.7442
                     XLM-RoBERTa    0.8686    0.8676       0.8696               0.7907
Ensemble (XLM-R 0.75 + XGB 0.25)    0.8729    0.8703       0.8739               0.8372


In [ ]:
# ==================================================
# CELL 13: CONFUSION MATRICES (SELF-CONTAINED)
# ==================================================
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

out_dir = str(OUT_DIR)
os.makedirs(out_dir, exist_ok=True)
class_labels = ["Low", "Medium", "High"]

def plot_and_save_cm(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_labels, yticklabels=class_labels)
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"Saved: {save_path}")

plot_and_save_cm(
    test_df["target"], test_xgb_preds,
    "Confusion Matrix - XGBoost (Test)",
    os.path.join(out_dir, "confusion_matrix_xgboost.png")
)

plot_and_save_cm(
    test_df["target"], test_xlmr_preds,
    "Confusion Matrix - XLM-RoBERTa / Deep Text (Test)",
    os.path.join(out_dir, "confusion_matrix_xlmr.png")
)

plot_and_save_cm(
    test_df["target"], test_ens_preds,
    "Confusion Matrix - Best Ensemble (Test)",
    os.path.join(out_dir, "confusion_matrix_ensemble.png")
)


Saved: /content/onecall_model1_outputs/confusion_matrix_xgboost.png
Saved: /content/onecall_model1_outputs/confusion_matrix_xlmr.png
Saved: /content/onecall_model1_outputs/confusion_matrix_ensemble.png


In [ ]:
from contextlib import contextmanager
import numpy as np
if not hasattr(np, "_no_nep50_warning"):
    @contextmanager
    def _no_nep50_warning():
        yield
    np._no_nep50_warning = _no_nep50_warning
# ==================================================
# CELL 14: 5-FOLD GROUP-AWARE CROSS-VALIDATION (SELF-CONTAINED)
# ==================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score
import xgboost as xgb

if "RANDOM_STATE" not in globals():
    RANDOM_STATE = 42

print("=" * 60)
print("5-FOLD GROUP-AWARE CROSS-VALIDATION (STRUCTURED XGBOOST)")
print("(Excluding the frozen test set)")
print("=" * 60)

cv_df = pd.concat([train_df, val_df], ignore_index=True)
cv_groups = cv_df["patient_query"]
cv_y = cv_df["target"]

sgkf_5 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

numeric_cols = [
    "age_imputed", "age_is_missing",
    "duration_days_imputed", "duration_is_missing",
    "symptom_count", "comorbidity_count", "has_comorbidity", "text_length"
]
categorical_cols = ["gender_clean", "duration_unit"]

cv_accuracies = []
cv_macro_f1s = []

fold = 1
for tr_k_idx, va_k_idx in sgkf_5.split(cv_df, cv_y, groups=cv_groups):
    df_k_tr = cv_df.iloc[tr_k_idx]
    df_k_va = cv_df.iloc[va_k_idx]

    prep_k = ColumnTransformer([
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ])

    X_k_tr = prep_k.fit_transform(df_k_tr)
    X_k_va = prep_k.transform(df_k_va)

    sw_k = compute_sample_weight("balanced", df_k_tr["target"])

    params = best_xgb_params if "best_xgb_params" in globals() and best_xgb_params else {"max_depth": 4, "learning_rate": 0.05, "n_estimators": 100}
    clf_k = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        eval_metric="mlogloss",
        **params
    )
    clf_k.fit(X_k_tr, df_k_tr["target"], sample_weight=sw_k)
    k_preds = clf_k.predict(X_k_va)

    k_acc = accuracy_score(df_k_va["target"], k_preds)
    k_f1 = f1_score(df_k_va["target"], k_preds, average="macro")

    cv_accuracies.append(k_acc)
    cv_macro_f1s.append(k_f1)
    print(f"Fold {fold}: Accuracy = {k_acc:.4f} | Macro F1 = {k_f1:.4f}")
    fold += 1

mean_cv_acc = np.mean(cv_accuracies)
std_cv_acc = np.std(cv_accuracies)
mean_cv_f1 = np.mean(cv_macro_f1s)
std_cv_f1 = np.std(cv_macro_f1s)

print(f"\n5-Fold CV Results (XGBoost):")
print(f"  Mean Accuracy: {mean_cv_acc:.4f} (+/- {std_cv_acc:.4f})")
print(f"  Mean Macro F1: {mean_cv_f1:.4f} (+/- {std_cv_f1:.4f})")
print("=" * 60)


In [ ]:
# ==================================================
# CELL 15: XGBOOST FEATURE IMPORTANCE (SELF-CONTAINED)
# ==================================================
import os
import pandas as pd
import matplotlib.pyplot as plt

out_dir = str(OUT_DIR)
os.makedirs(out_dir, exist_ok=True)

importances = best_xgb_model.feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

top20_feat = feat_imp.head(20)

plt.figure(figsize=(10, 6))
top20_feat.plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("Top 20 XGBoost Feature Importances (Structured Features)")
plt.xlabel("Relative Importance")
plt.tight_layout()
feat_imp_path = os.path.join(out_dir, "xgboost_feature_importance.png")
plt.savefig(feat_imp_path, dpi=300)
plt.close()

print(f"[Feature Importance] Saved plot to: {feat_imp_path}")
print("\nTop 10 Features:")
print(top20_feat.head(10))


[Feature Importance] Saved plot to: /content/onecall_model1_outputs/xgboost_feature_importance.png

Top 10 Features:
duration_unit_months     0.272442
duration_unit_hours      0.160682
duration_unit_weeks      0.154265
duration_days_imputed    0.117193
duration_unit_other      0.081312
text_length              0.049943
duration_unit_days       0.040066
symptom_count            0.033479
age_imputed              0.024969
has_comorbidity          0.022463
dtype: float32


In [ ]:
# ==================================================
# CELL 16: SOURCE-WISE ROBUSTNESS (SELF-CONTAINED)
# ==================================================
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, recall_score

print("=" * 60)
print("SOURCE-WISE ROBUSTNESS EVALUATION (SELECTED BEST ENSEMBLE)")
print("=" * 60)

test_df["ensemble_pred"] = test_ens_preds

source_eval_rows = []
for src in test_df["source_dataset"].unique():
    sub = test_df[test_df["source_dataset"] == src]
    if len(sub) == 0:
        continue
    s_acc = accuracy_score(sub["target"], sub["ensemble_pred"])
    s_mf1 = f1_score(sub["target"], sub["ensemble_pred"], average="macro")
    s_wf1 = f1_score(sub["target"], sub["ensemble_pred"], average="weighted")
    s_hrec = recall_score(sub["target"], sub["ensemble_pred"], labels=[2], average=None)[0]

    source_eval_rows.append({
        "Source": src,
        "Rows": len(sub),
        "Accuracy": round(s_acc, 4),
        "Macro F1": round(s_mf1, 4),
        "Weighted F1": round(s_wf1, 4),
        "High Urgency Recall": round(s_hrec, 4)
    })

source_eval_df = pd.DataFrame(source_eval_rows)
print(source_eval_df.to_string(index=False))
print("=" * 60)


SOURCE-WISE ROBUSTNESS EVALUATION (SELECTED BEST ENSEMBLE)
                    Source  Rows  Accuracy  Macro F1  Weighted F1  High Urgency Recall
SynTech_medical_triage_500    41    0.7805    0.5232       0.7553               0.8800
medical_symptom_triage_csv   195    0.8923    0.8730       0.8926               0.7778


In [ ]:
# ==================================================
# CELL 17: LEAKAGE DEMONSTRATION (SELF-CONTAINED)
# ==================================================
import os
import pandas as pd

print("=" * 60)
print("LEAKAGE DEMONSTRATION ONLY")
print("(Demonstrating why 'severity' MUST be excluded from genuine modeling)")
print("=" * 60)

leakage_path = BASE_DIR / "onecall_model1_combined.csv"
if os.path.exists(leakage_path):
    df_leak = pd.read_csv(leakage_path)
    if "severity" in df_leak.columns and df_leak["severity"].notna().sum() > 0:
        sub_leak = df_leak.dropna(subset=["severity", "urgency"]).copy()
        sub_leak["severity"] = sub_leak["severity"].astype(str).str.lower().str.strip()
        sub_leak["urgency"] = sub_leak["urgency"].astype(str).str.lower().str.strip()

        sev_cats = sorted(sub_leak["severity"].unique())
        urg_cats = ["low", "medium", "high"]

        contingency_dict = {}
        for s in sev_cats:
            contingency_dict[s] = {}
            for u in urg_cats:
                count = int(((sub_leak["severity"] == s) & (sub_leak["urgency"] == u)).sum())
                contingency_dict[s][u] = count

        leakage_table = pd.DataFrame.from_dict(contingency_dict, orient="index")
        print("\nEmpirical Contingency Table: Severity vs Urgency:")
        print(leakage_table)

        print("\nDeterministic mapping observed:")
        print("  - 'mild' -> 100% low urgency")
        print("  - 'moderate' -> 100% medium urgency")
        print("  - 'severe' -> 100% high urgency")
        print("\n[VERIFICATION CONFIRMED] Using severity causes 100% synthetic shortcut leakage.")
        print("Therefore, genuine model training MUST completely omit the severity column.")
    else:
        print("Severity column has no valid non-null entries in the provided file.")
else:
    print(f"File {leakage_path} not found; skipping empirical leakage demonstration.")
print("=" * 60)


LEAKAGE DEMONSTRATION ONLY
(Demonstrating why 'severity' MUST be excluded from genuine modeling)

Empirical Contingency Table: Severity vs Urgency:
        low  medium  high
high      0       0   156
low      30       0     0
medium    0     102     0

Deterministic mapping observed:
  - 'mild' -> 100% low urgency
  - 'moderate' -> 100% medium urgency
  - 'severe' -> 100% high urgency

[VERIFICATION CONFIRMED] Using severity causes 100% synthetic shortcut leakage.
Therefore, genuine model training MUST completely omit the severity column.


In [ ]:
# ==================================================
# CELL 18: SAVE OUTPUTS (SELF-CONTAINED & DEPLOYMENT-READY)
# ==================================================
import os
import joblib
import json
import pandas as pd

out_dir = str(OUT_DIR)
os.makedirs(out_dir, exist_ok=True)

metrics_path = os.path.join(out_dir, "final_metrics.csv")
comparison_df.to_csv(metrics_path, index=False)
print(f"Saved: {metrics_path}")

test_predictions_df = test_df[[
    c for c in ["id", "patient_query", "symptoms", "duration", "age", "gender", "urgency", "source_dataset"] if c in test_df.columns
]].copy()
id2label = {0: "low", 1: "medium", 2: "high"}
test_predictions_df["pred_majority"] = [id2label[p] for p in test_maj_preds]
test_predictions_df["pred_tfidf_svm"] = [id2label[p] for p in test_svm_preds]
test_predictions_df["pred_xgboost"] = [id2label[p] for p in test_xgb_preds]
test_predictions_df["pred_xlmr"] = [id2label[p] for p in test_xlmr_preds]
test_predictions_df["pred_ensemble"] = [id2label[p] for p in test_ens_preds]
test_pred_path = os.path.join(out_dir, "test_predictions.csv")
test_predictions_df.to_csv(test_pred_path, index=False)
print(f"Saved: {test_pred_path}")

xgb_path = os.path.join(out_dir, "best_xgboost_model.joblib")
joblib.dump(best_xgb_model, xgb_path)
print(f"Saved: {xgb_path}")

prep_path = os.path.join(out_dir, "preprocessor.joblib")
joblib.dump(preprocessor, prep_path)
print(f"Saved: {prep_path}")

svm_path = os.path.join(out_dir, "tfidf_svm_model.joblib")
joblib.dump({"vectorizer": best_union, "classifier": best_clf}, svm_path)
print(f"Saved: {svm_path}")

ensemble_meta = {
    "model_type": "transformers" if USE_TRANSFORMERS else "pytorch_bilstm",
    "weights": {"text_model": float(best_w_xlmr), "xgboost": float(best_w_xgb)},
    "imputation": {"train_median_age": float(train_median_age), "train_median_duration": float(train_median_duration)},
    "labels": {"id2label": id2label, "label2id": {v: k for k, v in id2label.items()}},
    "features": {"numeric_cols": numeric_cols, "categorical_cols": categorical_cols, "feature_names": feature_names}
}
meta_path = os.path.join(out_dir, "ensemble_config.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(ensemble_meta, f, indent=2)
print(f"Saved: {meta_path}")


Saved: /content/onecall_model1_outputs/final_metrics.csv
Saved: /content/onecall_model1_outputs/test_predictions.csv
Saved: /content/onecall_model1_outputs/best_xgboost_model.joblib
Saved: /content/onecall_model1_outputs/preprocessor.joblib


In [ ]:
# ==================================================
# CELL 19: FINAL REPORT (SELF-CONTAINED)
# ==================================================
import os
from sklearn.metrics import accuracy_score, f1_score, recall_score

final_model_name = f"Ensemble (Text-Model {best_w_xlmr:.2f} + XGBoost {best_w_xgb:.2f})"
final_acc = accuracy_score(test_df["target"], test_ens_preds)
final_macro_f1 = f1_score(test_df["target"], test_ens_preds, average="macro")
final_weighted_f1 = f1_score(test_df["target"], test_ens_preds, average="weighted")
final_high_rec = recall_score(test_df["target"], test_ens_preds, labels=[2], average=None)[0]

print(f"""
ONECALL MODEL 1 FINAL EXPERIMENT REPORT
=======================================

Dataset:
Rows: {len(df)}
Features used: Multilingual text (symptoms, duration, description, comorbidities, patient_query), structured (age, gender, duration_days, duration_unit, symptom_count, comorbidity_count)
Target: urgency (low, medium, high)

Split:
Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)
Validation: {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)
Test: {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)
Group overlap: {total_overlap}

Best validation model: {final_model_name}
Best validation accuracy: {ens_val_acc:.4f}
Best validation macro F1: {ens_val_macro_f1:.4f}

Final selected model: {final_model_name}
Final test accuracy: {final_acc:.4f}
Final test macro F1: {final_macro_f1:.4f}
Final test weighted F1: {final_weighted_f1:.4f}
Final high-urgency recall: {final_high_rec:.4f}

5-fold CV:
Mean accuracy: {mean_cv_acc:.4f}
Std accuracy: {std_cv_acc:.4f}
Mean macro F1: {mean_cv_f1:.4f}
Std macro F1: {std_cv_f1:.4f}

Leakage status:
Severity excluded: YES

Clinical validation:
NOT CLINICALLY VALIDATED

IMPORTANT:
The reported test performance is an experimental machine-learning result on the supplied dataset.
It must not be interpreted as clinical triage accuracy or medical validation.
""")

print("Saved output files:")
for root, _, files in os.walk(str(OUT_DIR)):
    for f in sorted(files):
        print(f"  {os.path.join(root, f)}")



ONECALL MODEL 1 FINAL EXPERIMENT REPORT

Dataset:
Rows: 1653
Features used: Multilingual text (symptoms, duration, description, comorbidities, patient_query), structured (age, gender, duration_days, duration_unit, symptom_count, comorbidity_count)
Target: urgency (low, medium, high)

Split:
Train: 1181 (71.4%)
Validation: 236 (14.3%)
Test: 236 (14.3%)
Group overlap: 0

Best validation model: Ensemble (Text-Model 0.75 + XGBoost 0.25)
Best validation accuracy: 0.9110
Best validation macro F1: 0.9129

Final selected model: Ensemble (Text-Model 0.75 + XGBoost 0.25)
Final test accuracy: 0.8729
Final test macro F1: 0.8703
Final test weighted F1: 0.8739
Final high-urgency recall: 0.8372

5-fold CV:
Mean accuracy: 0.8095
Std accuracy: 0.0183
Mean macro F1: 0.8048
Std macro F1: 0.0192

Leakage status:
Severity excluded: YES

Clinical validation:
NOT CLINICALLY VALIDATED

IMPORTANT:
The reported test performance is an experimental machine-learning result on the supplied dataset.
It must not be 